랭체인에서 체인이란?
Prompt와 LLM이 결합된 구조를 말한다.
사용자입력 (프롬프트)를 받아 LLM으로 응답을 생성하는 구조를 말한다.

추가 실습내용은 다음과 같다.
1. 멀티체인구성
2. Runnable 프로토콜을 구현한 객체를 조금 더 알아보자.

API발급받은 후 테스트 해보자.

In [3]:
!pip install -q langchain langchain-openai

In [4]:
# from google.colab import userdata
# import os

# os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_AI")

import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")


OpenAI API Key:  ········


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

result = llm.invoke("랭체인이란?")
print(result.content)

랭체인(LLM Chain)은 대규모 언어 모델(LLM)과 여러 가지 도구 또는 서비스 사이의 연계를 통해 복잡한 작업을 수행할 수 있도록 하는 프레임워크나 구조를 의미합니다. 이는 자연어 처리(NLP) 작업을 효율적으로 처리하기 위해 언어 모델의 기능과 다양한 외부 데이터 소스, API, 데이터베이스 등을 결합하여 활용하는 방식입니다.

랭체인의 주요 특징은 다음과 같습니다:

1. **모듈화**: 다양한 구성 요소(예: 데이터 수집, 데이터 전처리, 모델 추론, 후처리 등)를 모듈화하여 필요에 따라 조합할 수 있습니다.

2. **유연성**: 사용자는 특정 작업이나 응용 프로그램에 맞춰 랭체인을 설계하고 조정할 수 있습니다.

3. **강화된 기능**: 단순한 텍스트 생성 또는 분류 작업을 넘어서, 외부 정보와의 연계를 통해 더 복잡하고 다양한 기능을 수행할 수 있습니다.

4. **자동화**: 반복적인 작업을 자동화하여 생산성을 높이고, 보다 효율적으로 작업을 수행할 수 있게 도와줍니다.

예를 들어, 랭체인을 사용하면 고객 지원 챗봇이 FAQ 데이터베이스와 연결되어 자동으로 정보를 검색하고, 사용자의 질문에 실시간으로 응답할 수 있게 할 수 있습니다. 

랭체인은 다양한 산업 분야에서 활용될 수 있으며, 데이터 기반 의사결정 지원, 고객 서비스, 콘텐츠 생성 등 여러 분야에서 그 가능성을 보여주고 있습니다.


이제 단일체인을 구성해 보자 <br>
아까는 Prompt + Model 이였다면 이번에는
Prompt + Model + parser의 형태로 단일 체인을 만들어 볼 것이다.<br>

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatPromptTemplate.from_messages은 [ ]안에 역할지정 및 여러 메시지를 넣고 싶을때 사용한다.

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an computerscience teacher in middel school."),
    ("user", "Answer the question: {input}")
])

llm = ChatOpenAI(model="gpt-4o-mini")

# LLM의 복잡한 응답객체에서 우리가 원하는 깔끔한 문자열만 받을수 있게 도와주는 메서드
output_parser = StrOutputParser()

# LCEL chaining ((LangChain Expression Language) 문법)
# LangChain에서는 특별히 앞 단계의 출력을 다음 단계의 입력으로 연결하는 의미로 오버로딩(overload) 되어있음
chain = prompt | llm | output_parser

# invoke - 한번 실행후 결과를 반환하는 함수 (실행함수)
# chain 호출 - dict가 인자의 자료형임
chain.invoke({"input": "랭체인이란?"})

'랭체인(LLM Chain)은 대형 언어 모델(LLM: Large Language Model)을 활용하여 다양한 작업을 수행하고, 이들 작업을 서로 연결하여 복잡한 프로세스를 자동화하는 시스템입니다. 기본적으로 랭체인은 여러 개의 순차적 또는 병렬 작업을 구성하여, 사용자가 원하는 결과를 효과적으로 도출할 수 있도록 돕습니다.\n\n예를 들어, 랭체인을 사용하면 다음과 같은 작업을 수행할 수 있습니다:\n\n1. **질문 분석**: 사용자의 질문을 이해하고 분석합니다.\n2. **정보 검색**: 관련 정보를 데이터베이스나 외부 소스에서 검색합니다.\n3. **응답 생성**: 검색된 정보를 바탕으로 자연스러운 언어로 응답을 생성합니다.\n4. **후처리**: 생성된 응답을 검토하고 필요에 따라 수정합니다.\n\n랭체인은 특히 챗봇, 고객 지원 시스템, 콘텐츠 생성 등 다양한 분야에서 응용될 수 있습니다. 이를 통해 사용자는 보다 효율적이고 일관된 경험을 제공받을 수 있습니다.'

지금까지 본 것이 단일체인이라면<br>
지금 부터는 멀티체인을 보자.<br>
멀티체인은 말 그래도 여러개의 체인을 연결하는 구조를 말한다.

세개의 체인을 연결해보자. <br>
chain1 : "LLM 기초" -> 객관식 문제 3문제 생성 <br>
chain2 : 문제를 입력받아 정답과 해설을 생성 <br>
chain3 : 정답/해설을 분석해 문제의 난이도를 분류 <br>

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# prompt 정의
# 역할지정 안하고 단일메시지 보낼때는 from_message 말고 from_template 사용하면 됨

prompt = ChatPromptTemplate.from_template(
    """너는 고등학교 선생님이야.
    주제:{topic}
    위 주제에 대한 4지선다 객관식 문제를 3개 만들어줘
    문제:
    보기:
    """
)

prompt2 = ChatPromptTemplate.from_template(
    """
    너는 출판사의 해설작성 전문가야
    다음 문제들의 정답과 해설을 작성해줘
    {problems}
    정답:
    해설:
    """
)

prompt3 = ChatPromptTemplate.from_template(
    """
    다음 문제와 해설을 읽고 난이도를 분류해 주세요
    분류: 쉬움, 보통, 어려움
    {answer}
    """
)


llm = ChatOpenAI(model="gpt-4o-mini")

topic ="AI 이론"

# chain1 = prompt | llm | StrOutputParser()
# chain2 = prompt2 | llm | StrOutputParser()
# chain3= prompt3 | llm | StrOutputParser()

# # 문제생성
# problem = chain1.invoke({"topic": topic})
# print(f'생성된문제 {problem}')

# # 정답 및 해설
# answer = chain2.invoke({"problems": problem})
# print(f'정답 및 해설 {answer}')

# # 난이도분류
# difficulty = chain3.invoke({"answer": answer})
# print(f'난이도분류 {difficulty}')


# 코딩 컨벤션을 인라인 형식으로도 가능 (예제)
problem = (prompt | llm | StrOutputParser()).invoke({"topic": topic})
answer = (prompt2 | llm | StrOutputParser()).invoke({"problems": problem})
difficulty = (prompt3 | llm | StrOutputParser()).invoke({"answer": answer})

print(answer)
print("=============================")
print(difficulty)


**문제 1**: 인공지능의 기본 개념 중 "기계가 인간처럼 행동할 수 있도록 만드는 기술"을 무엇이라고 하는가?  
**보기**:  
A) 기계 학습 (Machine Learning)  
B) 인공 신경망 (Artificial Neural Network)  
C) 자율 주행 (Autonomous Driving)  
D) 인공지능 (Artificial Intelligence)  

**정답**: D) 인공지능 (Artificial Intelligence)  

**해설**:  
인공지능(Artificial Intelligence, AI)은 기계가 인간처럼 사고하고 행동할 수 있도록 하는 기술을 지칭합니다. AI는 다양한 알고리즘과 기술을 사용하여 다양한 작업을 수행하는 능력을 갖추며, 사람과 유사한 결정을 내리거나 행동을 할 수 있는 시스템을 개발하는 데 중점을 둡니다. 따라서 "기계가 인간처럼 행동할 수 있도록 만드는 기술"을 가장 정확하게 설명하는 용어는 인공지능입니다.

---

**문제 2**: 기계 학습의 한 가지 방법으로, 데이터를 통해 패턴이나 규칙을 학습하고 예측 모델을 만드는 기술은 무엇인가?  
**보기**:  
A) 지능형 에이전트 (Intelligent Agent)  
B) 강화 학습 (Reinforcement Learning)  
C) 감독 학습 (Supervised Learning)  
D) 자연어 처리 (Natural Language Processing)  

**정답**: C) 감독 학습 (Supervised Learning)  

**해설**:  
감독 학습(Supervised Learning)은 주어진 데이터셋을 사용하여 입력과 출력 간의 관계를 학습하는 학습 방법입니다. 이 방식에서는 레이블이 부착된 데이터, 즉 정답이 알려진 데이터를 이용해 알고리즘이 패턴을 인식하고, 이를 통해 새로운 데이터에 대한 예측을 수행합니다. 이는 기계 학습의 핵심적인 접근 방식 중 하나로, 주로 분류 및 회귀 문제에 사용됩니다. 따라서 데이터에서

* "프롬프트+LLM+파서”가 각각의 단위의 모듈이고 (Runnable 컴포넌트)<br> 

* invoke는 실행/연결 지점이라고 생각하면 가장 이해하기 쉽다. <br>

* 각 체인은 독립적으로 정의되어야 한다. 그리고 invoke 호출 시 이전 체인의 출력값을 입력으로 넣으면 된다. <br>

* 이렇게 하면 단계별로 중간 결과 확인이 가능 + 디버깅에 용이하다


LangChain을 호출하고 결과를 반환 할때 invoke 메소드를 사용했다.<br>
이러한 invoke 메소드를 "Runnable" 프로토콜 이라고 부른다. <br>
"Runnable" 프로토콜로는 invoke, batch, ainvoke 실행 메소드 들이있다 <br>
* batch는 입력 리스트에 대해 체인을 호출하고 결과를 리스트로 반환한다 <br>
* stream은 모델이 토큰을 생성하는 즉시 바로바로 흘려보내준다. 따라서 chatbot처럼 한글자씩 타이핑 되는 느낌을 주고 싶을때 사용하거나 긴 텍스트 요약을 토큰반위로 받아서 실시간으로 ui에 띄울떄 사용한다 <br>
* ainvoke는 단일 입력 실행을 비동기(async/await) 방식으로 지원

    * abatch -> 비동기로 여러 입력을 동시에 실행, 입력 순서대로 결과를 한번에 반환
    * ainvoke -> 비동기 실행 (결과는 한 번에 반환)
    * astream ->  비동기 스트리밍 실행 (결과를 청크 단위로 흘려줌)



In [14]:
# batch 메소드 사용예시

# 1. 컴포넌트 정의
prompt= ChatPromptTemplate.from_template("중학생이 이해하도록 {topic}에 대해서 간단하게 설명해주세요")

model = ChatOpenAI(model="gpt-4o-mini")

output_parser = StrOutputParser()

# 2. 컴포넌트 연결
chain = prompt | model | output_parser

# 3. batch 메소드 사용
topic = ["랭체인","RAG","Fine-tuning"]
result= chain.batch(topic)
for i in range(len(topic)):
    print(f'{topic[i]}에 대한 설명:{result[i]}')
    print(f'*****************')


랭체인에 대한 설명:랭체인(Chain of Thought, CoT)은 인공지능 모델이 문제를 해결하거나 질문에 답할 때, 생각하는 과정을 단계별로 정리하여 보여주는 방법입니다. 이 방식은 모델이 단순히 정답만을 제시하는 것이 아니라, 그 답을 어떻게 도출했는지를 설명하도록 도와줍니다.

예를 들어, 수학 문제를 풀 때, 랭체인을 사용하면 모델이 다음과 같이 생각할 수 있습니다:

1. 문제를 이해합니다.
2. 필요한 정보를 정리합니다.
3. 해결 방법을 선택합니다.
4. 계산을 수행합니다.
5. 답을 제시합니다.

이런 식으로 단계별로 사고 과정을 보여줌으로써, 사람들도 모델의 생각을 더 잘 이해할 수 있게 됩니다. 랭체인은 특히 복잡한 문제를 풀거나, 사람들이 질문을 했을 때 그에 대한 답을 잘 설명해야 할 때 유용합니다.
*****************
RAG에 대한 설명:RAG는 "Retrieval-Augmented Generation"의 약자로, 정보를 검색하고 생성하는 방식을 결합한 기술입니다. 쉽게 설명하자면, RAG는 질문을 받으면 먼저 관련된 정보를 검색한 다음, 그 정보를 기반으로 답변을 생성하는 시스템이에요.

예를 들어, 너가 "지구의 기온은 왜 변할까?"라는 질문을 한다면, RAG는 먼저 인터넷이나 데이터베이스에서 지구의 기온 변화에 대한 정보를 찾아보고, 그 정보를 바탕으로 쉽게 이해할 수 있는 답변을 만들어주는 방식이에요. 

즉, RAG는 단순히 알고 있는 대답을 하는 것이 아니라, 필요한 정보를 찾아보아서 더 정확하고 풍부한 답변을 제공해주는 기술이라고 생각하면 돼요!
*****************
Fine-tuning에 대한 설명:Fine-tuning은 이미 학습된 모델을 특별한 작업에 맞게 추가로 학습시키는 과정을 말해요. 예를 들어, 큰 데이터로 일반적인 언어 이해 모델을 처음 만들었을 때, 그 모델은 많은 일반적인 정보를 알고 있지만, 특정한 분야(예: 의학, 법률)에 대한 정보는 부족할 수 있어요. 

이럴 때, 그 모델

In [20]:
# stream 메소드 사용 예시

stream=chain.stream({"topic":"허깅페이스"}) # 청크 단위로 쪼개서 출력

for i in stream:
  print(i,end='',flush=True) # 버퍼를 비워 출력을 바로 화면에 띄움


허깅페이스(Hugging Face)는 인공지능, 특히 자연어 처리(NLP) 분야에서 매우 유명한 회사이자 플랫폼입니다. 자연어 처리는 사람들이 사용하는 언어를 컴퓨터가 이해하고 처리하는 기술을 말해요.

허깅페이스의 가장 잘 알려진 제품 중 하나는 "Transformers"라는 오픈소스 라이브러리입니다. 이 라이브러리는 여러 종류의 언어 모델(예: BERT, GPT 등)을 쉽게 사용할 수 있게 도와줘요. 즉, 사람들의 말을 이해하거나 문장을 생성하는 AI를 만드는 데 필요한 도구들이 모여 있는 곳이에요.

허깅페이스는 코딩을 잘 몰라도 쉽게 AI 모델을 사용할 수 있는 인터페이스도 제공해서, 많은 개발자와 연구자들이 편리하게 활용하고 있습니다. 그래서 허깅페이스는 AI와 관련된 다양한 프로젝트를 진행하는 사람들에게 매우 유용한 자원이 되고 있어요. 

쉽게 말해, 허깅페이스는 언어를 이해하는 AI를 만들기 위한 도구와 자원을 제공하는 곳이라고 생각하면 됩니다!

In [21]:
# ainvoke 예시

import asyncio                         # 파이썬 내장 비동기 라이브러리
from asyncio.tasks import as_completed # 비동기 작업을 "완료되는 순서대로" 처리하기 위한 함수
import nest_asyncio                    # 코랩, 주피터노트북에서 비동기 작업시 추가적으로 필요

nest_asyncio.apply()                   # (코랩/주피터에서만 필요, 비동기 실행 전에 1번만 선언)

async def run_async():                 # 비동기 함수 선언

    task=[
        chain.ainvoke({"topic":"랭체인"}),
        chain.ainvoke({"topic":"RAG"}),
        chain.ainvoke({"topic":"Fine-tuning"})
    ]

    for i in asyncio.as_completed(task): # 작업이 끝나는 순서대로 반환
        print(await i)                   # 비동기 작업 하나씩 await해서 결과로 출력
        print(f'*****************')

asyncio.run(run_async())  # 비동기 함수 실행할 때

Fine-tuning은 이미 학습된 모델을 조금 더 특정한 과제나 데이터에 맞게 조정하는 과정이에요. 예를 들어, 이미 기본적인 영어를 이해할 수 있는 인공지능이 있다고 해보세요. 이 인공지능을 특정 주제인 '코딩'에 대해 더 잘 이해하도록 하고 싶다면, 코딩 관련 자료로 다시 학습시키는 거죠. 

이 과정은 마치 학생이 이미 공부한 내용을 바탕으로 특정 과목을 더 깊이 배우는 것과 비슷해요. 처음부터 다시 배우는 것보다 더 빠르고 효율적으로 원하는 지식을 얻을 수 있어요. 그래서 Fine-tuning은 효과적인 방법이라고 볼 수 있죠!
*****************
RAG는 "Retrieval-Augmented Generation"의 약자로, 텍스트를 생성하는 인공지능 모델이 정보를 더 잘 제공하기 위해 사용하는 방법 중 하나예요. 간단하게 설명하자면, RAG는 다음과 같은 과정을 거쳐요:

1. **정보 검색:** 먼저, 질문이나 주제에 대한 관련 정보를 데이터베이스나 문서에서 검색해요. 이를 통해 모델은 더 많은 정보를 얻게 돼요.

2. **텍스트 생성:** 그 다음, 검색한 정보를 바탕으로 답변을 만들어 내요. 이 과정에서 모델은 사람의 자연스러운 언어처럼 글을 생성해요.

예를 들어, "유기농이란 무엇인가?"라는 질문을 한다면, RAG 모델은 유기농에 대한 여러 정보를 찾아내고, 그 정보를 바탕으로 유기농의 정의나 장점 등을 설명해 줄 수 있어요.

결국 RAG는 정보 검색과 텍스트 생성을 결합하여 더 정확하고 유용한 답변을 제공하는 방법이라고 생각하면 돼요!
*****************
랭체인(LLama Chain)은 자연어 처리(NLP)와 인공지능(AI) 분야에서 사용되는 도구입니다. 쉽게 설명하자면, 랭체인은 다양한 언어 모델이나 AI 기능을 연결하여 더 복잡한 작업을 수행하는 시스템입니다.

중학생 수준에서 이해할 수 있도록 예를 들어 설명해볼게요:

1. **AI와 대화**: 우리가 AI와 대화할 때, 질문을 하면 AI가 대답을 해주죠

여러번 실행해 보면 응답받은 순서대로 출력이 될 것입니다.

이제 다시 돌아가서 Rag에 대해서 살펴보겠습니다.